# SASRec+ Maximum Sequence Length

Use the team's shared data loader and original-style HR/NDCG evaluator. Start with **smoke** mode, then switch to **full**. Smoke results only show that the pipeline runs.

Before running: import this notebook into Kaggle; attach the private input containing `cmpe257_kaggle_code.zip` and `dataset.pkl`; enable Internet for dependencies and a GPU for full training. This notebook uses one GPU. No GitHub token or push is needed.

Read `docs/KAGGLE_TRAINING.md` in the repo for the full walkthrough. The notebook launches Python modules from the attached source ZIP; the notebook alone is not enough.

Select ml-100k as the `DATASET` in section 1. Each dataset uses its own weights and outputs. Start a fresh kernel when changing the dataset or code bundle. Multiple attached `dataset.pkl` files are resolved by their recorded checksum.


## 1. Load the shared code and dataset paths


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, sys, zipfile

# Attach the PRIVATE Kaggle dataset containing these two files.
# Leave as None to discover a single matching file; set explicit paths if needed.
CODE_ZIP = None  # e.g. '/kaggle/input/cmpe257-training/cmpe257_kaggle_code.zip'
CODE_DIR = None  # Optional extracted folder containing bundle_manifest.json
DATASET = 'ml-100k'  # 'baby', 'beauty', or 'ml-100k'
DATA_FILE = None  # e.g. '/kaggle/input/cmpe257-training/dataset.pkl'
INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

def attached_file(name, override):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(INPUT_ROOT.rglob(name))
    if len(matches) != 1:
        raise ValueError(f'Expected one {name}; found {matches}. Set its path above.')
    return matches[0]

REPO = WORK_ROOT / 'CMPE257_ML'
REPO.mkdir(parents=True, exist_ok=True)
zip_matches = sorted(INPUT_ROOT.rglob('cmpe257_kaggle_code.zip'))
archive = None
if CODE_DIR is None and (CODE_ZIP is not None or zip_matches):
    archive = attached_file('cmpe257_kaggle_code.zip', CODE_ZIP)
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            if not (REPO / member).resolve().is_relative_to(REPO.resolve()):
                raise ValueError('Unsafe archive member')
        z.extractall(REPO)
else:
    # Kaggle may unpack the uploaded ZIP before attaching it to a notebook.
    manifest_path = (Path(CODE_DIR) / 'bundle_manifest.json' if CODE_DIR
                     else attached_file('bundle_manifest.json', None))
    source_dir = manifest_path.parent.resolve()
    manifest = json.loads(manifest_path.read_text())
    for name in [*manifest['files'], 'bundle_manifest.json']:
        source, target = (source_dir / name).resolve(), (REPO / name).resolve()
        if not source.is_relative_to(source_dir) or not target.is_relative_to(REPO.resolve()):
            raise ValueError('Unsafe manifest path')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, target)
manifest = json.loads((REPO / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    file = (REPO / name).resolve()
    if not file.is_relative_to(REPO.resolve()):
        raise ValueError('Unsafe manifest path')
    assert hashlib.sha256(file.read_bytes()).hexdigest() == expected, name
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source branch:', manifest['branch'])
print('Base commit:', manifest['base_commit'])
if archive:
    print('Source ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
else:
    print('Loaded Kaggle-extracted source; all manifest hashes verified.')
from src.data.catalog import find_dataset, dataset_spec
DATASET_PATH = find_dataset(INPUT_ROOT, DATASET, DATA_FILE)
ARCHIVE_ROOT = WORK_ROOT if DATASET == 'baby' else WORK_ROOT / 'exports' / DATASET
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATASET, DATASET_PATH)


## 2. Install dependencies
If Kaggle requests a session restart after installation, restart and rerun from cell 1. Do not change package versions independently without recording the change.


In [ ]:
import subprocess
# Keep Kaggle's CUDA-enabled torch if it satisfies the shared requirements.
# Internet must be enabled for missing dependencies. Run before importing models.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade-strategy', 'only-if-needed',
                '-r', str(REPO / 'requirements-shared.txt')], check=True)


In [ ]:
import torch
from src.data import load_dataset, SequenceDataset
from src.data.load import dataset_summary
from src.training.kaggle import make_config, train, load_for_inference, predict_scores
from src.utils.reproducibility import seed_everything

print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    torch.set_num_threads(1)
    print('CPU is sufficient for smoke mode. Enable a GPU before full training.')
# Import every required model now, so dependency errors happen before training.
from sasrec import SASRec
from model import Att_Diffuse_model


## 3. Choose a run
Keep `RUN_MODE = "smoke"` for the first test. For a real experiment change it to `"full"`, choose a new `RUN_NAME`, and keep the agreed shared settings. `FINALIZE = False` saves progress without test evaluation when splitting training across sessions.


In [ ]:
RUN_MODE = 'full'  # 'full' uses all users in the selected dataset, hidden size 128, up to 500 epochs
RUN_NAME = 'sasrec_' + RUN_MODE + '_seed42'
FINALIZE = True
RESUME_CHECKPOINT = None  # Path to this model's latest.pt from a saved previous run
cfg = make_config('sasrec', RUN_MODE, dataset=DATASET)
# Example only, after agreeing with the team: cfg['train_batch_size'] = 128
# Keep evaluation_batch_size=512 for full runs, even if training batch size changes.
RUNS_ROOT = WORK_ROOT / 'runs'
if DATASET != 'baby':
    RUNS_ROOT /= DATASET
RUN_DIR = RUNS_ROOT / RUN_NAME
print(json.dumps(cfg, indent=2))
data = load_dataset(DATASET_PATH, cfg['dataset_sha256'])
print(dataset_summary(data))


## Train the model for lengths 10 and 25 across 5 seeds. Total of 10 runs

In [ ]:
import csv, statistics, hashlib, json, shutil

LENGTHS = [10, 25]
SEEDS = [42, 43, 44, 45, 46]
metrics = ['HR@5', 'HR@10', 'HR@20', 'NDCG@5', 'NDCG@10', 'NDCG@20']
OUT = WORK_ROOT / 'history_length' / DATASET            # small result files
MODELS = WORK_ROOT / 'history_length_models' / DATASET  # model weights (keep out of Git)
OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
all_summary = []

for L in LENGTHS:
    runs_dir = WORK_ROOT / 'runs' / DATASET / f'len{L}'
    # 1. Train every seed at this history length
    for s in SEEDS:
        cfg = make_config('sasrec', RUN_MODE, dataset=DATASET)
        cfg['seed'] = s
        cfg['max_len'] = L
        print('Training max_len', L, 'seed', s)
        train(cfg, DATASET_PATH,
              runs_dir / f'sasrec_{RUN_MODE}_len{L}_seed{s}' / 'sasrec',
              finalize=True)
    # 2. Collect and save the results for this history length
    out = OUT / f'len{L}'
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for s in SEEDS:
        folder = runs_dir / f'sasrec_{RUN_MODE}_len{L}_seed{s}' / 'sasrec'
        config = json.loads((folder / 'config.json').read_text())
        report = json.loads((folder / 'results.json').read_text())
        assert config['max_len'] == L and config['seed'] == s and config['mode'] == RUN_MODE, folder
        assert report['test']['protocol'] == 'adrec_upstream', folder
        for name in ['config.json', 'results.json', 'history.json', 'environment.json', 'requirements-lock.txt']:
            shutil.copyfile(folder / name, out / f'sasrec_len{L}_seed{s}_{name}')
        hashes = {n: hashlib.sha256((folder / n).read_bytes()).hexdigest() for n in ['best.pt', 'latest.pt']}
        (out / f'sasrec_len{L}_seed{s}_checkpoints.json').write_text(json.dumps(
            {'dataset': DATASET, 'max_len': L, 'seed': s, 'location': str(folder), 'files': hashes}, indent=2))
        shutil.make_archive(str(MODELS / f'sasrec_len{L}_seed{s}'), 'zip', root_dir=folder.parent)
        rows.append(dict(model='SASRec+', dataset=DATASET, max_len=L, seed=s,
                         best_epoch=report['best_epoch'], epochs_completed=report['epochs_completed'],
                         **{k: report['test']['metrics'][k] for k in metrics}))
    common = dict(model='SASRec+', dataset=DATASET, max_len=L, n_runs=len(rows), seeds='42-46')
    summary = [dict(common, statistic='mean', **{k: round(statistics.mean(r[k] for r in rows), 4) for k in metrics}),
               dict(common, statistic='std (sample, n-1)', **{k: round(statistics.stdev(r[k] for r in rows), 4) for k in metrics})]
    all_summary += summary
    for fname, vals in [(f'sasrec_{DATASET}_len{L}_runs.csv', rows), (f'sasrec_{DATASET}_len{L}_summary.csv', summary)]:
        with (out / fname).open('w', newline='') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(vals[0]))
            writer.writeheader(); writer.writerows(vals)
    print('Finished max_len', L)

with (OUT / f'sasrec_{DATASET}_history_length_summary.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(all_summary[0]))
    writer.writeheader(); writer.writerows(all_summary)
print('Results zip:', shutil.make_archive(str(WORK_ROOT / 'history_length_results'), 'zip', root_dir=OUT))
print('Model files folder:', MODELS)

## Create runs csv

In [ ]:
with (OUT / f'sasrec_{DATASET}_history_length_summary.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(all_summary[0]))
    writer.writeheader(); writer.writerows(all_summary)

rows_all = []
for L in LENGTHS:
    with (OUT / f'len{L}' / f'sasrec_{DATASET}_len{L}_runs.csv').open() as handle:
        rows_all += list(csv.DictReader(handle))
with (OUT / f'sasrec_{DATASET}_history_length_runs.csv').open('w', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows_all[0]))
    writer.writeheader(); writer.writerows(rows_all)

print('Results zip:', shutil.make_archive(str(WORK_ROOT / 'history_length_results'), 'zip', root_dir=OUT))
print('Model files folder:', MODELS)

## Print metrics

In [ ]:
for row in all_summary:
    print('max_len', row['max_len'], '|', row['statistic'],
          '| HR@20', row['HR@20'], '| NDCG@20', row['NDCG@20'])